# 11. Challenge the explanation before writing the paper

We will edit a longer motion sequence, keep observation support paired, inspect geometric damage, and identify the evidence needed for a neurological interpretation.

## Run this notebook on its own

From `experiments/aaai`, run `uv sync`, then `uv run jupyter lab`. Select the Python environment created here. Python 3.12, PyTorch, pandas, scikit-learn, matplotlib, python-dotenv, the notebook libraries, and the local `ambient` package are declared in `pyproject.toml`. Each notebook imports and loads everything it needs.

All configuration and credentials remain in the repository root `.env`. Set `AAAI_DEMO=1` for artificial arrays used to check the software, or `AAAI_DEMO=0` for available real poses. Restart the kernel after changing settings. Artificial examples have no medical meaning. The setup uses aliases for python-dotenv's actual function names so that `load_env(find_env())` works.

The video below provides real walking context in either mode. It is separate from any artificial example. These research notebooks do not launch a large download or full experiment without an explicit command.

## Historical tutorial and current study

This notebook preserves the earlier tutorial's code, observations and experimental scope. Its original result identities remain historical; they are not measurements of the new retained-predictor or U/J/A study. Start the current independent tutorial sequence at [Notebook 12](12_observation_population_and_video_context.ipynb), then continue through [Notebook 17](17_failure_analysis_and_manuscript_decision.ipynb). The [current protocol](docs/iteration02/protocol.md) and [current results](docs/iteration02/results.md) identify the verified new run. The [draft01 manuscript](docs/draft01/manuscript.md) remains the record of the completed earlier study.

The current primary question is normal versus the four requested condition families, with all other folders retained in a mandatory broad secondary analysis. It tests opposite-leg information beyond own-leg geometry and extra capacity. Uniform masking remains the reference, while J and A are declared motion-independent alternatives with matched budgets. Read the newer notebooks for their support rules and fixed evaluation banks. Do not combine old and new scores or denominators.

In [ ]:
from pathlib import Path
import os, sys, json
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'ambient').is_dir() and (p/'data/gavd').is_dir())
BASE=ROOT/'experiments/aaai'
os.chdir(ROOT)
from dotenv import load_dotenv as load_env, find_dotenv as find_env
load_env(find_env())
os.chdir(BASE)
if str(BASE) not in sys.path:sys.path.insert(0,str(BASE))
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display,Markdown
from aaai_gait.settings import Settings
from aaai_gait.study_data import build_study_manifest,make_outer_folds,fold_metadata
from aaai_gait.study_runner import prepare_study
from aaai_gait.media import show_video_context,plot_motion
settings=Settings();torch.set_num_threads(2)
fresh_manifest=build_study_manifest(ROOT,settings.pose_dir)
print('Mode:', 'SYNTHETIC SOFTWARE CHECK' if settings.demo else 'REAL GAVD, subject to quality exclusions')
print('Fresh annotations:',len(fresh_manifest),'sequences;',fresh_manifest.source_id.nunique(),'source videos')
video_row=show_video_context(fresh_manifest,frame_number_base=settings.frame_number_base)

## 1. Make an edit whose meaning is narrow and testable

Shift the entire left leg forward in time while keeping the rest of the body at its original time. Compare that edit with shifting the whole body by the same amount. The relative shift changes relationships between body regions; the whole-body shift changes the crop's phase. Both use interior samples from a longer sequence, with no circular wraparound or repeated boundary frames.

The leg retains its donor sequence's internal articulation, but its connection to the torso may become distorted. Measure that distortion explicitly. These edits test model sensitivity. They do not simulate stroke, Parkinson's disease, cerebral palsy, or muscle weakness.

![Paired timing and observation controls](docs/draft00/images/intervention_controls.svg)

In [ ]:
from aaai_gait.demo import synthetic_corpus
from aaai_gait.study_controls import paired_edits,common_intervention_rows
long_x,_=synthetic_corpus(seed=42,frames=40,sources_per_class=2)
edits,geometry=paired_edits(long_x[:2],shift=4,seed=42)
print('Artificial software demonstration only.')
display(geometry)
for name in ('original','relative_limb_shift','whole_body_shift'):
    plot_motion(edits[name][0],title=name);plt.show()
assert np.array_equal(edits['whole_body_shift'],long_x[:2,8:40])

## 2. Separate changed motion from degraded observation

Coordinate noise and observation dropout are separate controls. The learning mask still comes from the same uniform bank. It must not adapt to the edit. A prediction difference can otherwise come from a changed set of evaluated joints rather than changed movement.

For every version of a source, intersect valid target identities before comparing scores. An identity contains the window, original joint index, time patch, mask ID, and target type. A dropout experiment may leave too little support for a complete profile. Report that loss of coverage instead of replacing missing regions with zero.

The real control command uses development normal sources only, keeps the original crop's normalization fixed across edits, and applies the fitted reference without refitting it on edited data.

In [ ]:
from aaai_gait.context_prediction import ContextConfig,fit_context_models,score_context_models
small,small_meta=synthetic_corpus(seed=9,sources_per_class=6)
# First12 rows are artificial normal examples, six for each role.
mini,mini_history=fit_context_models(small[:6],small[6:12],
    config=ContextConfig(pretrain_updates=2,predictor_updates=2,batch_size=4,validation_batches=1),seed=42,
    train_source_ids=small_meta.source_id[:6].to_numpy(),validation_source_ids=small_meta.source_id[6:12].to_numpy(),
    train_labels=small_meta.label[:6].to_numpy(),validation_labels=small_meta.label[6:12].to_numpy())
raw={name:score_context_models(mini,values,mask_repeats=2) for name,values in edits.items()}
paired=common_intervention_rows(raw)
display(pd.DataFrame({'requested_rows':{k:len(v) for k,v in raw.items()},
                      'paired_rows':{k:len(v) for k,v in paired.items()}}))
keys=['window_index','joint','patch_index','mask_id','target_type']
for table in paired.values():
    assert table[keys].reset_index(drop=True).equals(paired['original'][keys].reset_index(drop=True))

## 3. Run the real development controls

After a completed real study, replace the path below with its printed run directory:

```bash
uv run python scripts/run_study_controls.py --run-dir artifacts/context_study/gavd/<run_id> --fold 0
```

The command chooses one eligible long window per development normal source using sequence and time order, not scores. Outputs contain paired profiles, source IDs, observation coverage, and geometry checks. Inspect the video and skeleton alongside these numbers. A relative shift producing an impossible pelvis or large torso changes cannot support a specific coordination explanation.

The teacher target is contextual. Editing other joints can change that target as well as predictor context. The coordinate branch is essential. The optional `--teacher-decomposition` flag now separates changes in predictor inputs from changes in the frozen teacher target. It evaluates four combinations on identical target support. For a relative left-leg edit, the diagnostic reads only unchanged arm and right-leg joints. A teacher-side change then demonstrates contextualization of the model target, not a change in those local joint measurements.

In [ ]:
kind='synthetic' if settings.demo else 'gavd'
latest=BASE/'artifacts/context_study'/kind/'latest.json'
run_dir=None
if latest.exists():
    pointer=json.loads(latest.read_text());run_dir=Path(pointer['run_dir'])
    if not run_dir.is_absolute():run_dir=ROOT/run_dir
    summary=json.loads((run_dir/'summary.json').read_text())
    assert bool(summary['synthetic'])==settings.demo
    print('Recorded artifact:',run_dir,'status:',summary['status'])
    display(summary)
else:
    print('No completed artifact in this mode. Run scripts/run_context_study.py first.')
    print('The calculations below teach the protocol without inventing results.')

In [ ]:
if run_dir is not None:
    control_dir=run_dir/'fold_0/development_controls'
    if (control_dir/'protocol.json').exists():
        display(json.loads((control_dir/'protocol.json').read_text()))
        display(pd.read_csv(control_dir/'geometry.csv').groupby('intervention').median(numeric_only=True))
    else:
        print('No real development-control result recorded for this run.')

## Inspect the contextual teacher directly

Let p0 and p1 denote predictor inputs before and after the edit. Let y0 and y1 denote teacher targets before and after it. Evaluate p0y0, p1y0, p0y1, and p1y1. The total change equals the predictor-only change, teacher-only change, and their interaction. This identity concerns raw losses. A nonlinear profile distance would require recomputing all four complete profiles against the same frozen reference.

Here we inspect only the unchanged right ankle while the left leg is shifted. The values below come from an artificial teaching sequence and a tiny model, so they establish software behavior only.

In [ ]:
from aaai_gait.teacher_decomposition import score_teacher_decomposition,decompose_teacher_losses
factors=score_teacher_decomposition(mini,edits['original'],edits['relative_limb_shift'],mask_repeats=2,joints=(28,))
deltas=decompose_teacher_losses(factors)
display(deltas.head())
print('Each factor retains the same target identities and common eligibility.')

## 4. Decide which manuscript claim survives

A strong contribution needs a reproducible improvement over simpler scores, evidence that learning matters, stability to reference and observation choices, and a plausible movement-level explanation. If only the latent target improves, discuss teacher contextualization as an alternative. If ridge performs equally well, emphasize the useful comparison rather than claiming a necessary JEPA innovation. If the untrained encoder matches the trained one, the experiment has not demonstrated an advantage from self-supervised learning.

A published skeleton anomaly baseline still needs a validated adaptation to these sixteen joints and missing observations. Read the [feasibility review](docs/published_baseline_feasibility.md). An inspired implementation or repaired forward pass is not a reproduced baseline.

![Mechanistic ambiguity](docs/draft00/images/mechanism_ambiguity.svg)

## 5. Keep the longer-term world model grounded

A physical gait model would separate a controller, muscle and body properties, ground contact, and the camera. Several combinations can produce similar two-dimensional motion. A useful research direction is to return a set of compatible explanations and identify which extra measurement would distinguish them. Force, EMG, measured strength, medication state, or clinical examination could provide additional evidence.

GAVD can help study visible motion and observation bias. It does not supply the ground truth needed to identify hidden biological causes. The current completion study is a controlled first step toward that program.

![A physical world-model research path](docs/draft00/images/physical_world_model.svg)

## Record what this experiment establishes

Keep the run protocol, source roles, exclusions, training curves, frozen reference parameters, and source scores together. A diagram explains an intended comparison. Only a completed real-data run supplies evidence for it. Read the [implemented protocol](docs/context_study_protocol.md), [current results](docs/context_study_results.md), and [review of the research direction](docs/draft00/README.md) before making a manuscript claim.